In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
import cv2
import numpy as np
import os
import random
from glob import glob
from tqdm.notebook import tqdm
from PIL import Image
from pathlib import Path

# ================= ⚙️ CẤU HÌNH TỐI ƯU =================
ROOT_FOLDER = '/kaggle/input/dataset-matching-new/dataset_finetune' 
SAVE_DIR = '/kaggle/working/checkpoints' 

# Config Training
BATCH_SIZE = 128       
LEARNING_RATE = 1e-4   # Hạ thấp LR để finetune ổn định
NUM_EPOCHS = 15
MARGIN = 0.5           # Margin 0.5 phù hợp cho vector đã normalize
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✓ Device: {DEVICE}")
print(f"✓ Data Root: {ROOT_FOLDER}")

# ==============================================================================
# COPY ĐOẠN NÀY ĐÈ VÀO PHẦN "1. AUGMENTATION CHIẾN LƯỢC" TRONG NOTEBOOK TRAINING
# ==============================================================================

class DroneDegradation(object):
    """
    Class này mô phỏng chất lượng kém của ảnh drone:
    Thu nhỏ ảnh xuống cực bé rồi phóng to lại để tạo hiệu ứng vỡ hạt (pixelation).
    """
    def __init__(self, scale_range=(0.1, 0.3)): 
        self.scale_range = scale_range

    def __call__(self, img):
        w, h = img.size
        # Chọn ngẫu nhiên tỉ lệ thu nhỏ (ví dụ còn 20% kích thước thật)
        scale = random.uniform(*self.scale_range)
        new_w, new_h = max(16, int(w * scale)), max(16, int(h * scale))
        
        # 1. Thu nhỏ (Mất thông tin chi tiết)
        img_small = img.resize((new_w, new_h), resample=Image.BILINEAR)
        # 2. Phóng to lại kích thước cũ (Tạo răng cưa/mờ)
        img_back = img_small.resize((w, h), resample=Image.NEAREST)
        return img_back

def get_transforms(is_train=True):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]
    
    if is_train:
        # A. Anchor (Ảnh mẫu): PHẢI "PHÁ HỦY" ĐỂ GIỐNG ẢNH DRONE
        transform_anchor = transforms.Compose([
            transforms.Resize((224, 224)),
            
            # --- CẢI TIẾN QUAN TRỌNG NHẤT ---
            # 80% cơ hội bị làm vỡ hạt, mờ nhòe giống camera drone chất lượng thấp
            transforms.RandomApply([DroneDegradation(scale_range=(0.1, 0.3))], p=0.8),
            
            # Thêm rung lắc (Blur) mạnh
            transforms.RandomApply([
                transforms.GaussianBlur(kernel_size=(5, 9), sigma=(1.5, 3.0))
            ], p=0.6),
            
            # Sai lệch màu sắc (Color Jitter)
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
            
            # Các biến đổi hình học cơ bản
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(90), # Xoay mọi góc
            
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        
        # B. Drone Crop (Ảnh thật): Giữ nguyên hoặc augment nhẹ
        transform_drone = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(30),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        return transform_anchor, transform_drone
    
    else:
        # Val/Test
        transform_valid = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])
        return transform_valid, transform_valid

# ================= 2. DATASET LOADER =================
class SiameseDroneDataset(Dataset):
    def __init__(self, root_folder, video_ids=None, transform=None, is_train=True):
        self.root_folder = root_folder
        self.is_train = is_train
        
        if transform is not None:
            self.transform_anchor, self.transform_drone = transform
        else:
            self.transform_anchor, self.transform_drone = None, None
        
        # Indexing Data
        self.anchors_map = {}   
        self.positives_map = {} 
        self.negatives = []     
        
        # Load Anchors
        for f in glob(os.path.join(root_folder, 'anchors', '*.jpg')):
            vid_id = os.path.basename(f).split('_ref_')[0] 
            self.anchors_map.setdefault(vid_id, []).append(f)
            
        # Load Positives
        for d in glob(os.path.join(root_folder, 'positives', '*')):
            vid_id = os.path.basename(d)
            imgs = glob(os.path.join(d, '*.jpg'))
            if imgs: self.positives_map[vid_id] = imgs
                
        # Load Negatives
        self.negatives = glob(os.path.join(root_folder, 'negatives', '**', '*.jpg'), recursive=True)
        
        # Filter valid IDs
        all_valid_ids = [v for v in self.positives_map.keys() if v in self.anchors_map]
        
        if video_ids is not None:
            self.valid_video_ids = [v for v in all_valid_ids if v in video_ids]
        else:
            self.valid_video_ids = all_valid_ids
            
        print(f"[{'TRAIN' if is_train else 'VAL'}] Classes: {len(self.valid_video_ids)} | Neg Pool: {len(self.negatives)}")

    def __len__(self):
        if not self.valid_video_ids: return 0
        return sum(len(self.positives_map[v]) for v in self.valid_video_ids)

    def _load_image(self, path):
        img = cv2.imread(path)
        if img is None: return Image.new('RGB', (224, 224))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return Image.fromarray(img)

    def __getitem__(self, idx):
        vid_id = random.choice(self.valid_video_ids)
        
        # 1. Anchor
        anchor_path = random.choice(self.anchors_map[vid_id])
        anchor_img = self._load_image(anchor_path)
        
        # 2. Positive
        pos_path = random.choice(self.positives_map[vid_id])
        positive_img = self._load_image(pos_path)
        
        # 3. Negative (70% Hard/Bg, 30% Other Class)
        if random.random() < 0.7 and self.negatives:
            neg_path = random.choice(self.negatives)
        else:
            other_ids = [v for v in self.valid_video_ids if v != vid_id]
            if other_ids:
                neg_id = random.choice(other_ids)
                neg_path = random.choice(self.positives_map[neg_id])
            else:
                neg_path = random.choice(self.negatives) if self.negatives else anchor_path
        negative_img = self._load_image(neg_path)

        if self.transform_anchor:
            return (self.transform_anchor(anchor_img), 
                    self.transform_drone(positive_img), 
                    self.transform_drone(negative_img))
            
        return anchor_img, positive_img, negative_img

# ================= 3. MODEL SIAMESE MOBILENET =================
class SiameseMobileNet(nn.Module):
    def __init__(self, pretrained=True, embedding_dim=576):
        super(SiameseMobileNet, self).__init__()
        if pretrained:
            full_model = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        else:
            full_model = mobilenet_v3_small(weights=None)
        
        self.features = full_model.features
        
        # Projection Head
        self.projection = nn.Sequential(
            nn.Linear(embedding_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128)
        )
        
    def forward_once(self, x):
        x = self.features(x)
        x = nn.functional.adaptive_avg_pool2d(x, (1, 1))
        x = x.flatten(1)
        x = self.projection(x)
        return torch.nn.functional.normalize(x, p=2, dim=1)
    
    def forward(self, anchor, positive, negative):
        return self.forward_once(anchor), self.forward_once(positive), self.forward_once(negative)

# ================= 4. TRAINING LOOP TỐI ƯU =================
class SiameseTrainer:
    def __init__(self, model, train_loader, val_loader, device, learning_rate, save_dir):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        
        # Optimizer: AdamW tốt hơn Adam thường
        self.optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        
        # Scheduler: Cosine Annealing để hội tụ mượt mà
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=NUM_EPOCHS)
        
        # Loss: Margin 0.5 cho vector normalized
        self.criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)
        self.scaler = GradScaler()        
        self.best_val_loss = float('inf')
    
    def calc_accuracy(self, a, p, n):
        # Tính khoảng cách Euclidean
        d_ap = (a - p).pow(2).sum(1)
        d_an = (a - n).pow(2).sum(1)
        # Accuracy: Tỉ lệ số lần khoảng cách AP < AN
        return (d_ap < d_an).float().mean()

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        total_acc = 0
        
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch} [Train]', leave=False)
        for anchor, positive, negative in pbar:
            anchor, positive, negative = anchor.to(self.device), positive.to(self.device), negative.to(self.device)
            
            self.optimizer.zero_grad()
            
            with autocast():
                emb_a, emb_p, emb_n = self.model(anchor, positive, negative)
                loss = self.criterion(emb_a, emb_p, emb_n)
                acc = self.calc_accuracy(emb_a, emb_p, emb_n)
            
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            total_loss += loss.item()
            total_acc += acc.item()
            
            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{acc.item():.2%}"})
        
        avg_loss = total_loss / len(self.train_loader)
        avg_acc = total_acc / len(self.train_loader)
        return avg_loss, avg_acc
    
    def validate(self):
        self.model.eval()
        total_loss = 0
        total_acc = 0
        
        with torch.no_grad():
            for anchor, positive, negative in self.val_loader:
                anchor, positive, negative = anchor.to(self.device), positive.to(self.device), negative.to(self.device)
                
                with autocast():
                    emb_a, emb_p, emb_n = self.model(anchor, positive, negative)
                    loss = self.criterion(emb_a, emb_p, emb_n)
                    acc = self.calc_accuracy(emb_a, emb_p, emb_n)
                    
                total_loss += loss.item()
                total_acc += acc.item()
                
        return total_loss / len(self.val_loader), total_acc / len(self.val_loader)
    
    def fit(self, epochs):
        print(f"🚀 Training start: {epochs} epochs | Batch: {BATCH_SIZE} | LR: {LEARNING_RATE}")
        
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = self.train_epoch(epoch)
            val_loss, val_acc = self.validate()
            
            self.scheduler.step()
            curr_lr = self.optimizer.param_groups[0]['lr']
            
            print(f"Epoch {epoch}: Train Loss={train_loss:.4f} (Acc={train_acc:.2%}) | "
                  f"Val Loss={val_loss:.4f} (Acc={val_acc:.2%}) | LR={curr_lr:.6f}")
            
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save(self.model.state_dict(), self.save_dir / 'siamese_mobilenet_best.pth')
                print(f"  >>> ⭐ Best model saved!")
        
        print("✅ Training Complete.")

# ================= 5. EXECUTION =================
def main():
    if not os.path.exists(ROOT_FOLDER):
        print(f"❌ Không tìm thấy thư mục {ROOT_FOLDER}")
        return

    # 1. Chia Train/Val
    all_video_ids = [os.path.basename(d) for d in glob(os.path.join(ROOT_FOLDER, 'positives', '*'))]
    if len(all_video_ids) < 2:
        train_ids, val_ids = all_video_ids, all_video_ids
    else:
        train_ids, val_ids = train_test_split(all_video_ids, test_size=0.2, random_state=42)

    # 2. Dataset
    train_dataset = SiameseDroneDataset(ROOT_FOLDER, train_ids, get_transforms(True), is_train=True)
    val_dataset = SiameseDroneDataset(ROOT_FOLDER, val_ids, get_transforms(False), is_train=False)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # 3. Train
    model = SiameseMobileNet(pretrained=True)
    trainer = SiameseTrainer(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        device=DEVICE, 
        learning_rate=LEARNING_RATE, 
        save_dir=SAVE_DIR
    )
    
    trainer.fit(epochs=NUM_EPOCHS)

if __name__ == "__main__":
    torch.cuda.empty_cache()
    main()

✓ Device: cuda
✓ Data Root: /kaggle/input/dataset-matching-new/dataset_finetune


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


[TRAIN] Classes: 11 | Neg Pool: 4044
[VAL] Classes: 3 | Neg Pool: 4044


100%|██████████| 9.83M/9.83M [00:00<00:00, 109MB/s]


🚀 Training start: 15 epochs | Batch: 128 | LR: 0.0001


...

Epoch 13: Train Loss=0.0579 (Acc=97.46%) | Val Loss=0.3239 (Acc=83.74%) | LR=0.000004
  >>> ⭐ Best model saved!


Epoch 14 [Train]:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 14: Train Loss=0.0540 (Acc=98.47%) | Val Loss=0.3834 (Acc=73.17%) | LR=0.000001


Epoch 15 [Train]:   0%|          | 0/7 [00:00<?, ?it/s]

Epoch 15: Train Loss=0.0551 (Acc=98.77%) | Val Loss=0.3065 (Acc=85.37%) | LR=0.000000
  >>> ⭐ Best model saved!
✅ Training Complete.
